<a href="https://colab.research.google.com/github/pradeep6kumar/SD_textual_inversion_purple_guidance/blob/aoc/purple_guidance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
from diffusers import StableDiffusionPipeline
import numpy as np
from PIL import Image
import os
from tqdm import tqdm
from huggingface_hub import hf_hub_download
import warnings

# Suppress symlink warnings
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = "1"


The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

In [2]:

# Create results directory
os.makedirs("results", exist_ok=True)

# Define styles and their seeds
styles = {
    "glitch": {
        "concept_url": "sd-concepts-library/001glitch-core",
        "seed": 42,
        "token": "<glitch-core>"
    },
    "roth": {
        "concept_url": "sd-concepts-library/2814-roth",
        "seed": 123,
        "token": "<2814-roth>"
    },
    "night": {
        "concept_url": "sd-concepts-library/4tnght",
        "seed": 456,
        "token": "<4tnght>"
    },
    "anime80s": {
        "concept_url": "sd-concepts-library/80s-anime-ai",
        "seed": 789,
        "token": "<80s-anime>"
    },
    "animeai": {
        "concept_url": "sd-concepts-library/80s-anime-ai-being",
        "seed": 1024,
        "token": "<80s-anime-being>"
    }
}


In [3]:


def load_base_model():
    """Load the base Stable Diffusion model with safety checker enabled"""
    return StableDiffusionPipeline.from_pretrained(
        "CompVis/stable-diffusion-v1-4",
        torch_dtype=torch.float16
    ).to("cuda")

def download_embedding(style_info):
    """Download embedding file from Hugging Face hub"""
    try:
        local_path = hf_hub_download(
            repo_id=style_info["concept_url"],
            filename="learned_embeds.bin",
            repo_type="model"
        )
        return local_path
    except Exception as e:
        raise Exception(f"Failed to download embedding: {str(e)}")

def generate_with_style(prompt, style_name, pipe):
    """Generate an image with a specific style"""
    try:
        style_info = styles[style_name]
        generator = torch.Generator("cuda").manual_seed(style_info['seed'])
        styled_prompt = f"{prompt} {style_info['token']}"
        print(f"Generating with prompt: {styled_prompt}")
        return pipe(
            styled_prompt,
            generator=generator,
            guidance_scale=7.5,  # Standard guidance scale
            num_inference_steps=50  # Standard number of steps
        ).images[0]
    except Exception as e:
        print(f"Error generating image for {style_name}: {e}")
        return None

def purple_loss(image):
    """Calculate purple loss for an image"""
    img_array = np.array(image)
    purple_target = np.array([128, 0, 128])
    diff = np.abs(img_array - purple_target).mean(axis=-1)
    return diff.mean()

def apply_purple_guidance(image, strength=0.5):
    """Apply purple guidance to an image"""
    img_array = np.array(image).astype(float)
    purple_mask = (img_array[:,:,0] > 100) & (img_array[:,:,2] > 100)
    img_array[purple_mask] = img_array[purple_mask] * (1 - strength) + np.array([128, 0, 128]) * strength
    return Image.fromarray(np.uint8(img_array.clip(0, 255)))


In [4]:

def main():
    # Load base model
    print("Loading base model...")
    pipe = load_base_model()

    # Load embeddings
    print("\nLoading style embeddings...")
    for style_name, info in styles.items():
        try:
            embedding_path = download_embedding(info)
            pipe.load_textual_inversion(embedding_path)
            print(f"Loaded {style_name} embedding")
        except Exception as e:
            print(f"Error loading {style_name}: {e}")
            continue

    # Base prompt to use for all generations
    base_prompt = "A serene mountain landscape with a lake at sunset"

    print("\nGenerating images for each style...")
    for style_name in tqdm(styles.keys()):
        # Generate original image
        original = generate_with_style(base_prompt, style_name, pipe)
        if original is None:
            print(f"Skipping {style_name} due to generation error")
            continue

        # Calculate and apply purple guidance
        original_loss = purple_loss(original)
        guided = apply_purple_guidance(original)
        guided_loss = purple_loss(guided)

        # Save results
        original.save(f"results/{style_name}_original.png")
        guided.save(f"results/{style_name}_purple_guided.png")

        # Print loss values
        print(f"\n{style_name}:")
        print(f"  Original purple loss: {original_loss:.2f}")
        print(f"  Guided purple loss: {guided_loss:.2f}")


In [ ]:
main()

Loading base model...


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model_index.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]

text_encoder%2Fconfig.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

...

unet%2Fconfig.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

tokenizer%2Ftokenizer_config.json:   0%|          | 0.00/806 [00:00<?, ?B/s]

diffusion_pytorch_model.safetensors:   0%|          | 0.00/3.44G [00:00<?, ?B/s]

vae%2Fconfig.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

diffusion_pytorch_model.safetensors:   0%|          | 0.00/335M [00:00<?, ?B/s]

### Loading the output results from 'results folder'

In [ ]:
import os
from PIL import Image
import matplotlib.pyplot as plt

# Update this to your own folder path
image_folder = r"./results"

# Grab all files that end with .jpg, .jpeg, or .png, etc.
valid_exts = ('.jpg', '.jpeg', '.png')
image_files = [f for f in os.listdir(image_folder) if f.lower().endswith(valid_exts)]

# Sort them if you’d like a consistent order (alphabetical, etc.)
image_files.sort()

# Display each image with its filename as the title
for filename in image_files:
    file_path = os.path.join(image_folder, filename)
    img = Image.open(file_path)

    plt.figure()             # Create a new figure for each image
    plt.imshow(img)          # Show the image
    plt.title(filename)      # Set the image filename as the title
    plt.axis('off')          # Hide the axes for a cleaner look
    plt.show()               # Render the figure
